# 🤖 Simple RAG Demo — HR Policy Assistant

**RAG** stands for **Retrieval-Augmented Generation**. In plain English:

1. We take a document (here, an HR policy handbook).
2. We chop it into small pieces and turn each piece into a list of numbers (an "embedding") that captures its meaning.
3. We store those pieces in a searchable database (a "vector store").
4. When a user asks a question, we **search** the database for the most relevant pieces, and hand them to an AI model to **generate** a final answer.

That's it. No magic — just search + a language model.

**What we use in this notebook:**
- 📄 Data: `data/hr_policy.txt` (a sample HR policy document)
- 🔢 Embeddings: **Jina AI**
- 🗄️ Vector store: **FAISS**
- 🧠 LLM: **Groq** (fast + free-tier friendly)
- 🕸️ Framework: **LangChain** (`create_agent`)

> Before running: make sure the notebook's kernel is set to the `ragenv` virtual environment (top-right corner of VS Code / Jupyter), and that a `.env` file with `GROQ_API_KEY` and `JINA_API_KEY` exists in this folder.

In [82]:
import os 
from dotenv import load_dotenv

In [83]:
# LLM MODEL - text as input / text as clear_output

# EMB MDOEL - text as ipnput / embedding as outptu

In [84]:
# langchain

from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter 

from langchain_community.embeddings import JinaEmbeddings


In [85]:
load_dotenv()

True

In [86]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """Rule 1: Always greet the customer with a polite hello.
Rule 2: Never give out internal server passwords.
Rule 3: Escalate to a human if the user gets angry."""

# Setup the smart splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=60,       # Maximum characters allowed per chunk
    chunk_overlap=10     # How many characters to repeat from the previous chunk
)

# Chop the text
chunks = splitter.split_text(text)

# Print the results
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: '{chunk}'")


Chunk 1: 'Rule 1: Always greet the customer with a polite hello.'
Chunk 2: 'Rule 2: Never give out internal server passwords.'
Chunk 3: 'Rule 3: Escalate to a human if the user gets angry.'


In [130]:
groq_key = os.getenv("GROQ_API_KEY")  # LLM 
jina_key = os.getenv("JINA_API_KEY")   # EMBEDDING MODEL

print("ENV VAR LOADED")

ENV VAR LOADED


LOAD THE TEXT

In [88]:
DATA_FILE_PATH = os.path.join("data" , "hr_policy.txt")


In [89]:
loader = TextLoader(DATA_FILE_PATH , encoding="utf-8")
documents = loader.load()

In [90]:
len(documents)

1

In [91]:
documents[0].metadata

{'source': 'data\\hr_policy.txt'}

LANGCHAIN DOCUMENT

PAGE CONTENT   - actual data 

METADATA  - you have info of your data 

In [92]:
print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: data\hr_policy.txt
Number of documents loaded: 1
Total characters in document: 2603

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


In [93]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [94]:
len(chunks)

9

NOW EACH SPILTED CHUNK IS A DOCUMENT - page content and metadata

In [95]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [96]:
print(chunks[2])

page_content='2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.' metadata={'source': 'data\\hr_policy.txt'}


In [97]:
print(chunks[6])

page_content='6. CODE OF CONDUCT
Employees are expected to maintain professionalism and respect in the workplace.
Harassment, discrimination, or any form of workplace misconduct will not be tolerated
and may result in disciplinary action, including termination.
All employees must complete an annual Code of Conduct training.' metadata={'source': 'data\\hr_policy.txt'}


EMBEDDING

In [98]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

In [99]:
embeddings_model.model_name

'jina-embeddings-v2-base-en'

In [100]:
embeddings_model

JinaEmbeddings(session=<requests.sessions.Session object at 0x00000195D0F97770>, model_name='jina-embeddings-v2-base-en', jina_api_key=None)

In [101]:
query_vector = embeddings_model.embed_query("What is LangChain?")

query_vector

[-0.00962079,
 -0.03576203,
 -0.00614623,
 -0.00571117,
 -0.02768769,
 0.01993371,
 -0.00105305,
 -0.02854793,
 0.06249061,
 0.07040081,
 -0.02841345,
 -0.04780927,
 -0.01529932,
 -0.00992187,
 -0.05072023,
 0.09596659,
 -0.01636127,
 -0.01124633,
 0.05107618,
 -0.03327032,
 0.00255549,
 -0.02981356,
 -0.04322135,
 -0.02895135,
 0.07470396,
 0.02174911,
 0.07038499,
 0.05853551,
 0.02928357,
 0.0218638,
 -0.01902799,
 -0.02199037,
 -0.03330196,
 0.00566049,
 0.00278785,
 -0.03365001,
 -0.01911754,
 -0.04297021,
 0.02452163,
 0.07486217,
 -0.04831553,
 -0.01027338,
 0.00664309,
 0.08210792,
 -0.04645663,
 -0.00460572,
 -0.01889748,
 0.02273393,
 -0.05072616,
 0.02377412,
 0.00210819,
 -0.04055957,
 0.03049383,
 -0.05992771,
 0.02606017,
 0.01649475,
 0.05625737,
 -0.04516726,
 -0.02222965,
 -0.0238374,
 -0.01881837,
 0.03407716,
 -0.02861121,
 0.01495622,
 0.01808569,
 0.02791907,
 -0.01657979,
 -0.08058916,
 -0.01703858,
 0.0062629,
 -0.01701633,
 0.0159707,
 0.02659411,
 0.05451712,
 

text -- emeddings - vector db


In [102]:
from langchain_community.vectorstores import FAISS 

In [103]:
vector_store = FAISS.from_documents(chunks , embeddings_model)

In [104]:
print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [105]:
test_query = "How many sick leaves employees get"

VECTOR SEARCH 

In [106]:
top_matches = vector_store.similarity_search(test_query , k=3)

In [107]:
top_matches

[Document(id='7eb29f39-2930-45f7-b51a-24dde242f38d', metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'),
 Document(id='c311ae4d-b575-43e3-ad2a-15f83b687cb5', metadata={'source': 'data\\hr_policy.txt'}, page_content='7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.'),
 Document(id='38326fed-9c6e-4a2e-ab28-0eb584a30a96', metadata={'source': 'data\\hr_policy.txt

In [108]:
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

--- Match 3 ---
4. NOTICE PERIOD
Employees who wish to resign must serve a notice period of 30 days.
During the probation period, the notice period is reduced to 15 days.
The company may waive the notice period at its discretion, with full and final se

END OF DATA INGESTING PIPELINE

TOOL

In [110]:
vector_store.similarity_search

<bound method FAISS.similarity_search of <langchain_community.vectorstores.faiss.FAISS object at 0x00000195F1A4F980>>

In [109]:
vector_store

In [111]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks


def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

LLM 

In [131]:
from langchain_groq import ChatGroq 


llm = ChatGroq(
    model = "groq:openai/gpt-oss-120b",
    temperature=0.5
)

In [132]:
llm.model_name

'groq:openai/gpt-oss-120b'

In [ ]:
# from langchain.chat_models import init_chat_model

# model = init_chat_model(
#     "groq:openai/gpt-oss-120b"
# )

In [ ]:
# model.model_name

'openai/gpt-oss-120b'

In [133]:
tr = llm.invoke("Hey what is the leave policy")

NotFoundError: Error code: 404 - {'error': {'message': 'The model `groq:openai/gpt-oss-120b` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [134]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b")
llm.invoke("Say hi").content


'Hi! 👋 How can I help you today?'

In [135]:
answer = llm.invoke("tell me about code of conduct")

In [137]:
print(answer.content)

## What Is a Code of Conduct?

A **Code of Conduct (CoC)** is a set of rules and expectations that guide the behavior of participants in a community, organization, event, or project. It’s the social contract that helps create a welcoming, respectful, and safe environment for everyone involved—whether you’re contributing to an open‑source project, attending a conference, or working in a workplace.

---

## Why It Matters

| Benefit | Why It Helps |
|---------|--------------|
| **Safety & Inclusion** | Protects people from harassment, discrimination, or unwanted behavior. |
| **Clarity** | Sets clear expectations, reducing ambiguity about “what’s OK.” |
| **Trust** | Builds confidence that the community will treat all members fairly. |
| **Accountability** | Provides a transparent process for reporting and addressing violations. |
| **Reputation** | Demonstrates professionalism and can attract more diverse participants. |

---

## Core Elements of a Strong Code of Conduct

| Section | Ty

In [153]:
answer.content

'## What Is a Code of Conduct?\n\nA **Code of Conduct (CoC)** is a set of rules and expectations that guide the behavior of participants in a community, organization, event, or project. It’s the social contract that helps create a welcoming, respectful, and safe environment for everyone involved—whether you’re contributing to an open‑source project, attending a conference, or working in a workplace.\n\n---\n\n## Why It Matters\n\n| Benefit | Why It Helps |\n|---------|--------------|\n| **Safety & Inclusion** | Protects people from harassment, discrimination, or unwanted behavior. |\n| **Clarity** | Sets clear expectations, reducing ambiguity about “what’s OK.” |\n| **Trust** | Builds confidence that the community will treat all members fairly. |\n| **Accountability** | Provides a transparent process for reporting and addressing violations. |\n| **Reputation** | Demonstrates professionalism and can attract more diverse participants. |\n\n---\n\n## Core Elements of a Strong Code of Cond

In [152]:

answer.response_metadata


{'token_usage': {'completion_tokens': 1600,
  'prompt_tokens': 77,
  'total_tokens': 1677,
  'completion_time': 1.7397253080000001,
  'completion_tokens_details': {'reasoning_tokens': 90},
  'prompt_time': 0.034838063,
  'prompt_tokens_details': None,
  'queue_time': 0.328400472,
  'total_time': 1.774563371},
 'model_name': 'openai/gpt-oss-20b',
 'system_fingerprint': 'fp_9340e7d14d',
 'service_tier': 'on_demand',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'groq'}

In [154]:
from langchain.agents import create_agent 

In [174]:
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    
    always reply in 2 lines , if you are not clear ask a questions to clarift from user
    
    Dont entertain any other questions which are off topic and if you dont understand ,
    tell that mail to acmecorp.com
    """,
    
)

In [171]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user",
                            "content": question}]})
    


    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [170]:
response = hr_assistant.invoke({"messages": [{"role": "user",
                            "content": "tell me about code of conduct"}]})

In [162]:
response

{'messages': [HumanMessage(content='tell me about code of conduct', additional_kwargs={}, response_metadata={}, id='1af3b96c-c560-4aba-9551-b7ad06a63a84'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "tell me about code of conduct". We need to use search_hr_policy tool to look up facts. Then answer. If not found, say I don\'t know. Also must not entertain any other question off topic. But this is within policy. So we will search.', 'tool_calls': [{'id': 'fc_9726200d-ca3b-4eab-9e79-de700290004f', 'function': {'arguments': '{"question":"code of conduct"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 233, 'total_tokens': 317, 'completion_time': 0.086867279, 'completion_tokens_details': {'reasoning_tokens': 58}, 'prompt_time': 0.011766168, 'prompt_tokens_details': None, 'queue_time': 0.315476538, 'total_time': 0.098633447}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerp

In [165]:
response["messages"][0].content

'tell me about code of conduct'

In [166]:
print(response["messages"][-1].content)

**Acme Crop – Code of Conduct**

- **Professionalism & Respect** – All employees must conduct themselves with professionalism and respect at all times.  
- **Harassment & Discrimination** – Any form of harassment or discrimination is strictly prohibited and may lead to disciplinary action, up to and including termination.  
- **Annual Training** – Every employee is required to complete the annual Code of Conduct training to stay up‑to‑date on expectations and policies.  

If you have any questions or need further clarification, feel free to ask!


In [167]:
print(ask_hr_assistant("tell me about PROBATION PERIOD"))

QUESTION: tell me about PROBATION PERIOD
------------------------------------------------------------
ANSWER: **Probation Period – Acme Crop HR Policy**

1. **Duration**  
   - All new employees undergo a **3‑month probation period** starting from their date of joining.

2. **Leave Eligibility During Probation**  
   - Employees are **not eligible for paid leave** while on probation.  
   - They may take **unpaid leave** in case of emergencies, but this requires **manager approval**.

3. **Performance Review**  
   - A performance review is conducted at the end of the probation period to determine whether employment is confirmed.

4. **Notice Period During Probation**  
   - If an employee wishes to resign during probation, the notice period is **15 days** (reduced from the standard 30 days).

5. **Settlement**  
   - The company may waive the notice period at its discretion, and a full and final settlement will be processed within **45 days** of the last working day.

*For any additio

In [172]:
print(ask_hr_assistant("tell me about that policy"))

QUESTION: tell me about that policy
------------------------------------------------------------
ANSWER: Could you clarify which policy you’re referring to?  
For example, leave, work‑from‑home, probation, or something else?

Could you clarify which policy you’re referring to?  
For example, leave, work‑from‑home, probation, or something else?


In [173]:
print(ask_hr_assistant("what did i ask"))

QUESTION: what did i ask
------------------------------------------------------------
ANSWER: You asked: “what did i ask.”  
Is there anything else you need help with?

You asked: “what did i ask.”  
Is there anything else you need help with?


In [185]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt=""" 
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know 
    instead of guessing.
    
    Always reply in 2 lines, if you are not clear ask a questions to clarify from user.
    
    Dont entertain any other questions which are off topic and if you dont understand,
    tell that mail to acmecorp.com.
    """,
    checkpointer=InMemorySaver(),
)

# Invoke with a thread_id to maintain memory across turns
config = {"configurable": {"thread_id": "user-session-1"}}




In [ ]:
response = hr_assistant.invoke(
    {"messages": [{"role": "user", "content": "What is the that policy?"}]},
    config=config,
)

In [179]:
response["messages"][-1].content

'Could you clarify which policy you’re interested in?  \nLet me know so I can look it up for you.'

In [189]:
response = hr_assistant.invoke(
    {"messages": [{"role": "user", "content": "How to make a coffee"}]},
    config=config,
)

In [196]:
response = hr_assistant.invoke(
    {"messages": [{"role": "user", "content": "what was my last question"}]},
    config=config,
)

In [197]:
response["messages"][-1].content

'I’m sorry, I can’t help with that.  \nPlease email acmecorp.com for assistance.'

In [200]:
response = model_response = hr_assistant.invoke(
    {"messages": [{"role": "user", "content": "What is the code of cunduct?"}]},
    config=config,
)

# Follow-up turns on the same thread reuse history automatically
follow_up = hr_assistant.invoke(
    {"messages": [{"role": "user", "content": "How many days do I get?"}]},
    config=config,
)

In [201]:
follow_up["messages"][-1].content

'I’m not sure which leave type you’re referring to—annual paid leave, sick leave, or another benefit.  \nCould you let me know which days you’re asking about?'